CONNECT

In [ ]:
# for a quicker set up ignore the step in the quickstart where you create a Snowflake environment, uncomment the below code and install the libraries provided
# there may be some errors about the datasets package but the lab should still run regardless
# %pip install snowflake pandas notebook scikit-learn cachetools pyarrow==10.0.1 snowflake-ml-python

In [3]:
import pandas as pd
from snowflake.snowpark import Session
from snowflake.snowpark.functions import *
from snowflake.snowpark.types import *

connection_parameters = {
    "account": "MF72391",
    "user": "SRIJA", 
    "host": "TLSOURR-MF72391.snowflakecomputing.com", # e.g. "sn00111.snowflakecomputing.com",
    "password": "Srijastar@1234",
    "role": "ACCOUNTADMIN",
    "warehouse": "SMALL_WH",
    "database":"HOL_DB",
    "schema":"PUBLIC"
    }
session = Session.builder.configs(connection_parameters).create()

In [4]:
# Loading from local CSV-files
city_udi_df = pd.read_csv('city_udi.csv')
humidity_df = pd.read_csv('humidity.csv')
maintenance_df = pd.read_csv('maintenance.csv')

LOAD

In [7]:
print(type(city_udi_df))
print(type(city_udi_df.reset_index()))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [11]:
session.get_current_warehouse()

'"SMALL_WH"'

In [16]:
# Check current role
print("Current role:", session.get_current_role())

# Check available roles
session.sql("SHOW ROLES").show()

Current role: "ACCOUNTADMIN"
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"created_on"                      |"name"                   |"is_default"  |"is_current"  |"is_inherited"  |"assigned_to_users"  |"granted_to_roles"  |"granted_roles"  |"owner"       |"comment"                                           |"is_from_organization_user_group"  |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-09-05 18:05:24.074000-07:00  |ACCOUNTADMIN             |Y             |Y             |N               |1                    |0              

In [17]:
session.use_role("ACCOUNTADMIN")
session.use_warehouse('"SMALL_WH"')

Failed to execute query [queryID: 01c6e4b9-3204-8088-0008-29060001ba26] use warehouse "SMALL_WH"
002043 (02000): SQL compilation error:
Object does not exist, or operation cannot be performed.


ProgrammingError: 002043 (02000): SQL compilation error:
Object does not exist, or operation cannot be performed.

In [13]:
# Convert to a list of dictionaries
records = city_udi_df.reset_index().to_dict(orient="records")

# Create Snowpark DataFrame and save directly
snowpark_df = session.create_dataframe(records)
snowpark_df.write.mode("overwrite").save_as_table("CITY_UDF")

SnowparkSQLException: (1304): 01c6e4b5-3204-8088-0008-29060001ba1e: 000606 (57P03): No active warehouse selected in the current session.  Select an active warehouse with the 'use warehouse' command.


In [12]:
snowpark_df = session.create_dataframe(city_udi_df.reset_index())
snowpark_df.write.mode("overwrite").save_as_table("CITY_UDF")

TypeError: create_dataframe() function only accepts data as a list, tuple or a pandas DataFrame.

In [10]:
# Upload to Snowflake
session.write_pandas(city_udi_df, table_name='CITY_UDF', auto_create_table=True, overwrite=True)
session.write_pandas(humidity_df, table_name='HUMIDITY', auto_create_table=True, overwrite=True)
session.write_pandas(maintenance_df, table_name='MAINTENANCE', auto_create_table=True, overwrite=True)

MissingDependencyError: Missing optional dependency: pandas